In [0]:
-- WITH filtered_hcp AS (
--     SELECT DISTINCT
--         npi_num__v AS hcp_npi,
--         formatted_name__v,
--         specialty_1__v,
--         vid__v AS hcp_vid
--     FROM com_edp_prd.com_raw.vod_hcp
--     WHERE specialty_1__v IN ('PA', 'PHM')
-- ),

-- ranked_vod_affiliations AS (
--     SELECT
--         a.hcp_npi,
--         a.formatted_name__v,
--         a.specialty_1__v,
--         c.vid__v AS hco_vid,
--         c.corporate_name__v AS hco_name,
--         c.npi_num__v AS hco_npi,
--         b.modified_date__v,
--         b.status_update_time__v,
--         ROW_NUMBER() OVER (
--             PARTITION BY a.hcp_npi
--             ORDER BY
--                 b.modified_date__v DESC NULLS LAST,
--                 b.status_update_time__v DESC NULLS LAST
--         ) AS rn
--     FROM filtered_hcp a
--     LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
--         ON a.hcp_vid = b.entity_vid__v
--        AND b.hierarchy_type__v = 'HCP_HCO'
--     LEFT JOIN com_edp_prd.com_raw.vod_hco c
--         ON b.parent_hco_vid__v = c.vid__v
--     WHERE b.parent_hco_status__v = 'A'
--       AND b.relationship_type__v = '7356'
-- )

-- SELECT
-- *

-- FROM ranked_vod_affiliations
-- WHERE rn = 1;

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent;

In [0]:
CREATE OR REPLACE TEMP VIEW temp_pharmacist_hcp_reporting_parent AS
SELECT
    t.*,
    CASE
        WHEN CAST(t.reporting_hco_npi AS STRING) IN (
            '1437365186','1235234535','1295789907','1114969169','1346297843',
            '1336245828','1104001858','1104819366','1811080526','1295820256',
            '1366515488','1912939703','1205935012','1750482022','1194787218',
            '1548212988','1891765178','1023105400','1649347469','1336495910',
            '1760476659','1235339227','1184649345','1578693321','1013143213',
            '1013062769','1518911338','1073053757','1467442749','1144548322',
            '1285174649','1659877280','1003063280','1275564098','1093894131',
            '1013924372','1609824010','1669429577','1033439732','1760480503',
            '1235148594','1750458485','1114924834','1083789630','1669462420',
            '1477643690','1649261462','1023188851','1043447253','1477549756',
            '1154302727','1083949382','1003961251','1235214834','1265694442',
            '1376544320','1679973364','1568596765','1003878539','1326092404',
            '1093808040','1598784555','1689747552','1164426896','1063702785',
            '1083630073','1235582925','1639370059','1275694184','1255577466',
            '1669683512','1932280666','1013924182','1225249865'
        ) THEN 'Tier 1'   --74/79

        WHEN CAST(t.reporting_hco_npi AS STRING) IN (
            '1043435902','1063617280','1245520386','1306013552','1407801640',
            '1740215219','1871540237','1003246471','1003819319','1003968579',
            '1013981554','1023494473','1073576740','1093728743','1134329923',
            '1134470156','1144313743','1154339588','1184612764','1255461075',
            '1396837951','1417946021','1447352836','1447355771','1467525790',
            '1477531580','1578568481','1669530069','1699720086','1821017880',
            '1922178789','1942685920','1952359986','1053635466','1265671739',
            '1811056898','1487306692','1538402995','1366281750','1982609442',
            '1023134657','1043397292','1154373843','1437119310','1528042884',
            '1073835567','1083781892','1154448769','1235250663','1376903351',
            '1295137404','1326332289','1376577247','1447233788','1467505073',
            '1568435477','1780676650','1255431987','1700914025','1700026077',
            '1497352934'
        ) THEN 'Tier 2'  --61/74

        WHEN CAST(t.reporting_hco_npi AS STRING) IN (
            '1003083445','1013071653','1407813660','1487844015','1871886366',
            '1013961093','1043354111','1043554967','1083711790','1134309305',
            '1164400131','1164474235','1215989249','1255748059','1275992240',
            '1285016758','1306145826','1396129524','1417061193','1417125642',
            '1417901521','1467544684','1497812721','1497850671','1518018191',
            '1578792271','1609869916','1609915164','1689953093','1710408265',
            '1720029333','1750381281','1801828421','1811944101','1285676544',
            '1841844099','1326558180','1144282583','1710072798'
        ) THEN 'Tier 3' --39/39

        ELSE NULL
    END AS reporting_hco_tier
FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent t;

In [0]:
SELECT COUNT(DISTINCT hcp_npi) AS hcps_identified_via_specialty
FROM temp_pharmacist_hcp_reporting_parent;

In [0]:
SELECT
    COUNT(DISTINCT hcp_npi) AS hcps_with_any_affiliated_hco,
    COUNT(DISTINCT CASE WHEN vod_hco_npi IS NOT NULL THEN hcp_npi END) AS hcps_with_vod_hco,
    COUNT(DISTINCT CASE WHEN komodo_hco_npi IS NOT NULL THEN hcp_npi END) AS hcps_with_komodo_hco
FROM temp_pharmacist_hcp_reporting_parent;

In [0]:
SELECT
    COUNT(DISTINCT hcp_npi) AS hcps_identified_via_specialty,

    COUNT(DISTINCT CASE
        WHEN vod_hco_npi IS NOT NULL OR komodo_hco_npi IS NOT NULL
        THEN hcp_npi
    END) AS hcps_with_any_affiliated_hco,

    COUNT(DISTINCT CASE
        WHEN vod_hco_npi IS NOT NULL
        THEN hcp_npi
    END) AS hcps_with_vod_hco,

    COUNT(DISTINCT CASE
        WHEN komodo_hco_npi IS NOT NULL
        THEN hcp_npi
    END) AS hcps_with_komodo_hco,

    COUNT(DISTINCT CASE
        WHEN reporting_hco_tier = 'Tier 1'
        THEN hcp_npi
    END) AS tier_1_hcps,

    COUNT(DISTINCT CASE
        WHEN reporting_hco_tier = 'Tier 2'
        THEN hcp_npi
    END) AS tier_2_hcps,

    COUNT(DISTINCT CASE
        WHEN reporting_hco_tier = 'Tier 3'
        THEN hcp_npi
    END) AS tier_3_hcps,

    COUNT(DISTINCT CASE
        WHEN reporting_hco_tier IN ('Tier 1', 'Tier 2', 'Tier 3')
        THEN hcp_npi
    END) AS total_tier_mapped_hcps
FROM temp_pharmacist_hcp_reporting_parent;

In [0]:
select reporting_hco_tier, count(distinct reporting_hco_npi) from temp_pharmacist_hcp_reporting_parent group by 1 

In [0]:
select count(distinct hcp_npi) from temp_pharmacist_hcp_reporting_parent where reporting_hco_tier in ('Tier 1', 'Tier 2','Tier 3')

In [0]:
SELECT DISTINCT
    t.hcp_npi,
    CONCAT(t.first_name__v, ' ', t.last_name__v) AS hcp_name,
    t.first_name__v,
    t.last_name__v,

    /* ================= HCP DETAILS ================= */
    v.postal_code_cda__v AS hcp_zip,
    z.state AS hcp_state,

    /* ✅ Specialty Fallback (Komodo → VOD) */
    COALESCE(kp_hcp.primary_specialty, v.specialty_1__v, '-') AS hcp_primary_specialty,
    COALESCE(kp_hcp.secondary_specialty, v.specialty_2__v, '-') AS hcp_secondary_specialty,

    t.hcp_specialty,

    /* ================= HCO DETAILS ================= */
    t.reporting_hco_npi,
    t.reporting_hco_name,

    /* Reordered / added columns */
    CASE 
        WHEN hco_vod.postal_code_cda__v IS NOT NULL THEN hco_vod.address_line_1__v
        ELSE kp.provider_address
    END AS hco_address,

    COALESCE(hco_vod.postal_code_cda__v, kp.provider_zip) AS hco_zip,
    hz.city AS hco_city,
    hz.state AS hco_state,
    hz.territory_name AS hco_territory,
    hz.region_name AS hco_region,

    t.vod_hco_npi,
    t.komodo_hco_npi,
    t.affiliation_source,
    t.reporting_hco_tier

FROM temp_pharmacist_hcp_reporting_parent t

/* ================= HCP - VOD ================= */
LEFT JOIN com_edp_prd.com_raw.vod_hcp v
    ON t.hcp_npi = v.npi_num__v

/* ================= HCP - KOMODO ================= */
LEFT JOIN com_edp_prd.com_raw.kom_providers kp_hcp
    ON t.hcp_npi = kp_hcp.npi
   AND kp_hcp.provider_type = 'INDIVIDUAL'

/* ================= HCP STATE ================= */
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON v.postal_code_cda__v = z.zipcode

/* ================= HCO - VOD ADDRESS ================= */
LEFT JOIN (
    SELECT
        h.npi_num__v,
        a.address_line_1__v,
        a.postal_code_cda__v,
        ROW_NUMBER() OVER (
            PARTITION BY h.npi_num__v
            ORDER BY a.modified_date__v DESC
        ) AS rn
    FROM com_edp_prd.com_raw.vod_hco h
    JOIN com_edp_prd.com_raw.vod_address a
        ON a.entity_vid__v = h.vid__v
       AND a.entity_type__v = 'HCO'
       AND a.record_state__v = 'VALID'
       AND a.address_status__v IN ('A','DS')
       AND a.address_verification_status__v NOT IN ('NS','U')
) hco_vod
    ON t.reporting_hco_npi = hco_vod.npi_num__v
   AND hco_vod.rn = 1

/* ================= HCO - KOMODO FALLBACK ================= */
LEFT JOIN com_edp_prd.com_raw.kom_providers kp
    ON t.reporting_hco_npi = kp.npi
   AND kp.provider_type = 'ORGANIZATION'

/* ================= HCO GEO ================= */
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping hz
    ON COALESCE(hco_vod.postal_code_cda__v, kp.provider_zip) = hz.zipcode

WHERE t.reporting_hco_tier IN ('Tier 1', 'Tier 2', 'Tier 3');

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.patient360;

## Appendix

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================================================
# Config
# =========================================================
TARGET_TABLE = "com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent"
TEMP_HCP_BASE = "com_edp_prd.cmpa_insights_internal_schema.tmp_pharmacist_hcp_base"

# Adaptive execution helps on large joins/shuffles
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Optional: if your environment supports this and it behaves well
# spark.conf.set("spark.sql.shuffle.partitions", "auto")


# =========================================================
# 1) Load only required columns
# =========================================================
vod_references = (
    spark.table("com_edp_prd.com_raw.vod_references")
    .select("reference_type", "name", "code")
)

vod_hcp = (
    spark.table("com_edp_prd.com_raw.vod_hcp")
    .select(
        "npi_num__v",
        "first_name__v",
        "last_name__v",
        "specialty_1__v",
        "vid__v"
    )
)

vod_parenthco = (
    spark.table("com_edp_prd.com_raw.vod_parenthco")
    .select(
        "entity_vid__v",
        "parent_hco_vid__v",
        "hierarchy_type__v",
        "parent_hco_status__v",
        "relationship_type__v",
        "modified_date__v",
        "status_update_time__v"
    )
)

vod_hco = (
    spark.table("com_edp_prd.com_raw.vod_hco")
    .select(
        "vid__v",
        "npi_num__v",
        "corporate_name__v"
    )
)

kom_providers = (
    spark.table("com_edp_prd.com_raw.kom_providers")
    .select(
        "npi",
        "provider_type",
        "hco_primary_npi",
        "organization_name",
        "provider_address",
        "provider_zip"
    )
)

vod_address = (
    spark.table("com_edp_prd.com_raw.vod_address")
    .select(
        "entity_vid__v",
        "entity_type__v",
        "record_state__v",
        "address_status__v",
        "address_verification_status__v",
        "address_line_1__v",
        "postal_code_cda__v",
        "modified_date__v"
    )
)

zip_map = (
    spark.table("com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping")
    .select("zipcode", "city", "state")
)


# =========================================================
# 2) Specialty reference
# Small lookup table -> broadcast
# =========================================================
specialty_ref = (
    vod_references
    .filter(
        (F.col("reference_type") == "Specialty") &
        (
            F.lower(F.col("name")).like("%clinical pharmacology%") |
            F.lower(F.col("name")).like("%pharmacology%") |
            F.lower(F.col("name")).like("%pharmacy specialty%") |
            F.lower(F.col("name")).like("%pharmaceutical medicine%")
        )
    )
    .select("code")
    .dropDuplicates()
)


# =========================================================
# 3) Build HCP base once
# Includes hcp_vid directly so we avoid re-reading vod_hcp later
# =========================================================
hcp_base = (
    vod_hcp.alias("h")
    .join(
        F.broadcast(specialty_ref).alias("s"),
        F.col("h.specialty_1__v") == F.col("s.code"),
        "inner"
    )
    .filter(F.col("h.npi_num__v").isNotNull())
    .select(
        F.col("h.npi_num__v").alias("hcp_npi"),
        F.col("h.first_name__v"),
        F.col("h.last_name__v"),
        F.col("h.specialty_1__v").alias("hcp_specialty"),
        F.col("h.vid__v").alias("hcp_vid")
    )
    .dropDuplicates(["hcp_npi", "hcp_vid"])
)

# =========================================================
# 4) Materialize HCP base as Delta temp table
# Serverless-safe replacement for persist()/cache()
# =========================================================
spark.sql(f"DROP TABLE IF EXISTS {TEMP_HCP_BASE}")

(
    hcp_base.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(TEMP_HCP_BASE)
)

hcp_base = spark.table(TEMP_HCP_BASE)


# =========================================================
# 5) VOD affiliation
# Filter parent HCO table before join/window
# =========================================================
vod_parenthco_filtered = (
    vod_parenthco
    .filter(
        (F.col("hierarchy_type__v") == "HCP_HCO") &
        (F.col("parent_hco_status__v") == "A") &
        (F.col("relationship_type__v") == "7356")
    )
)

vod_hco_slim = (
    vod_hco
    .select(
        F.col("vid__v").alias("hco_vid"),
        F.col("npi_num__v").alias("vod_hco_npi"),
        F.col("corporate_name__v").alias("vod_hco_name")
    )
)

vod_ranked = (
    hcp_base.alias("a")
    .join(
        vod_parenthco_filtered.alias("b"),
        F.col("a.hcp_vid") == F.col("b.entity_vid__v"),
        "left"
    )
    .join(
        vod_hco_slim.alias("c"),
        F.col("b.parent_hco_vid__v") == F.col("c.hco_vid"),
        "left"
    )
    .select(
        F.col("a.hcp_npi"),
        F.col("c.vod_hco_npi"),
        F.col("c.vod_hco_name"),
        F.col("b.modified_date__v"),
        F.col("b.status_update_time__v")
    )
)

vod_window = Window.partitionBy("hcp_npi").orderBy(
    F.col("modified_date__v").desc_nulls_last(),
    F.col("status_update_time__v").desc_nulls_last()
)

vod_final = (
    vod_ranked
    .withColumn("rn", F.row_number().over(vod_window))
    .filter(F.col("rn") == 1)
    .select(
        "hcp_npi",
        "vod_hco_npi",
        "vod_hco_name"
    )
)


# =========================================================
# 6) Komodo fallback
# Split individual and organization views once
# =========================================================
kom_individual = (
    kom_providers
    .filter(F.col("provider_type") == "INDIVIDUAL")
    .select(
        F.col("npi").alias("hcp_npi"),
        F.col("hco_primary_npi").alias("komodo_hco_npi")
    )
)

kom_org = (
    kom_providers
    .filter(F.col("provider_type") == "ORGANIZATION")
    .select(
        F.col("npi").alias("org_npi"),
        F.col("organization_name").alias("komodo_hco_name"),
        F.col("provider_address"),
        F.col("provider_zip")
    )
)

komodo = (
    hcp_base.alias("a")
    .join(
        kom_individual.alias("b"),
        "hcp_npi",
        "left"
    )
    .join(
        kom_org.select(
            "org_npi",
            "komodo_hco_name"
        ).alias("c"),
        F.col("b.komodo_hco_npi") == F.col("c.org_npi"),
        "left"
    )
    .select(
        F.col("a.hcp_npi"),
        F.col("b.komodo_hco_npi"),
        F.col("c.komodo_hco_name")
    )
)


# =========================================================
# 7) Base output
# Prioritize VOD, then Komodo, else '-'
# =========================================================
base_output = (
    hcp_base.alias("a")
    .join(vod_final.alias("v"), "hcp_npi", "left")
    .join(komodo.alias("k"), "hcp_npi", "left")
    .select(
        F.col("hcp_npi"),
        F.col("first_name__v"),
        F.col("last_name__v"),
        F.col("hcp_specialty"),
        F.coalesce(
            F.col("v.vod_hco_npi"),
            F.col("k.komodo_hco_npi"),
            F.lit("-")
        ).alias("reporting_hco_npi"),
        F.coalesce(
            F.col("v.vod_hco_name"),
            F.col("k.komodo_hco_name"),
            F.lit("-")
        ).alias("reporting_hco_name"),
        F.when(F.col("v.vod_hco_npi").isNotNull(), F.lit("VOD"))
         .when(F.col("k.komodo_hco_npi").isNotNull(), F.lit("Komodo"))
         .otherwise(F.lit("None"))
         .alias("affiliation_source")
    )
)


# =========================================================
# 8) VOD HCO address
# Filter hard before windowing
# =========================================================
hco_vod_join = (
    vod_hco
    .select(
        F.col("vid__v").alias("hco_vid"),
        F.col("npi_num__v").alias("hco_npi")
    )
)

vod_address_filtered = (
    vod_address
    .filter(
        (F.col("entity_type__v") == "HCO") &
        (F.col("record_state__v") == "VALID") &
        (F.col("address_status__v").isin("A", "DS")) &
        (~F.col("address_verification_status__v").isin("NS", "U"))
    )
)

hco_addr_ranked = (
    hco_vod_join.alias("h")
    .join(
        vod_address_filtered.alias("a"),
        F.col("h.hco_vid") == F.col("a.entity_vid__v"),
        "inner"
    )
    .select(
        F.col("h.hco_npi"),
        F.col("a.address_line_1__v"),
        F.col("a.postal_code_cda__v"),
        F.col("a.modified_date__v")
    )
)

addr_window = Window.partitionBy("hco_npi").orderBy(
    F.col("modified_date__v").desc_nulls_last()
)

hco_vod_address = (
    hco_addr_ranked
    .withColumn("rn", F.row_number().over(addr_window))
    .filter(F.col("rn") == 1)
    .select(
        "hco_npi",
        "address_line_1__v",
        "postal_code_cda__v"
    )
)


# =========================================================
# 9) Final enrichment
# Address fallback:
#   - if VOD zip exists, use VOD address
#   - else use Komodo org address
# =========================================================
final_output = (
    base_output.alias("b")
    .join(
        hco_vod_address.alias("v"),
        F.col("b.reporting_hco_npi") == F.col("v.hco_npi"),
        "left"
    )
    .join(
        kom_org.alias("kp"),
        F.col("b.reporting_hco_npi") == F.col("kp.org_npi"),
        "left"
    )
    .withColumn(
        "reporting_parent_address",
        F.when(
            F.col("v.postal_code_cda__v").isNotNull(),
            F.col("v.address_line_1__v")
        ).otherwise(F.col("kp.provider_address"))
    )
    .withColumn(
        "reporting_parent_zip",
        F.coalesce(F.col("v.postal_code_cda__v"), F.col("kp.provider_zip"))
    )
    .join(
        F.broadcast(zip_map).alias("z"),
        F.col("reporting_parent_zip") == F.col("z.zipcode"),
        "left"
    )
    .select(
        F.col("b.hcp_npi"),
        F.col("b.first_name__v"),
        F.col("b.last_name__v"),
        F.col("b.hcp_specialty"),
        F.col("b.reporting_hco_npi"),
        F.col("b.reporting_hco_name"),
        F.col("b.affiliation_source"),
        F.col("reporting_parent_address"),
        F.col("reporting_parent_zip"),
        F.col("z.city").alias("reporting_parent_city"),
        F.col("z.state").alias("reporting_parent_state")
    )
)


# =========================================================
# 10) Optional validation checks
# Comment out if you do not want actions before write
# =========================================================
# duplicate_hcp_count = (
#     final_output
#     .groupBy("hcp_npi")
#     .count()
#     .filter(F.col("count") > 1)
#     .count()
# )
# print(f"Duplicate HCP NPIs in final output: {duplicate_hcp_count}")


# =========================================================
# 11) Write final table
# =========================================================
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

(
    final_output.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)


# =========================================================
# 12) Cleanup temp table
# =========================================================
spark.sql(f"DROP TABLE IF EXISTS {TEMP_HCP_BASE}")